In [58]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import normalize
import datetime
from tabulate import tabulate
results = []

In [59]:
vector_size = 200
window = 10
epochs = 10

In [60]:
datapath = "../../dataset_v2/chessBig_tx.data"

df = pd.read_csv(datapath, sep=';')
df = df.drop(columns=['ID'])

columns_to_use = df.columns[df.columns != 'CLASS']



In [61]:
df['processed_text'] = df[columns_to_use].apply(lambda row: ' '.join(row.astype(str)), axis=1)

df['processed_text'] = df['processed_text'].apply(lambda x: x.split())


In [62]:
model = Word2Vec(sentences=df['processed_text'], vector_size=vector_size, window=window, min_count=1, workers=4, sg=1, epochs=epochs)

In [63]:
model_path = f"../../models/chessBig.model"
model.save(model_path)

In [64]:

model_dw = Word2Vec.load(model_path)

def get_average_vector(words, model_dw):
    vectors = []
    for word in words:
        if word in model_dw.wv:
            vectors.append(model_dw.wv[word])
    if vectors:
        return normalize(np.mean(vectors, axis=0).reshape(1, -1))[0]
    else:
        return np.zeros(model_dw.vector_size)
    



X = np.array([get_average_vector(text, model_dw) for text in df['processed_text']])
print(X.shape)

(28056, 200)


In [68]:
n_clusters = 300
start_time = datetime.datetime.now()
kmeans = KMeans(n_clusters=n_clusters)
kmeans.fit(X)
end_time = datetime.datetime.now()
running_time = end_time - start_time


y_kmeans = kmeans.predict(X)


real_labels_numeric = df['CLASS']

nmi = normalized_mutual_info_score(real_labels_numeric, y_kmeans)

results.append({
    "n_clusters": n_clusters,
    "vector_size": vector_size, 
    "window": window, 
    "epochs": epochs, 
    "NMI": nmi, 
    "running_time": running_time
})
print(tabulate(results, headers="keys", tablefmt="fancy_grid"))

╒══════════════╤═══════════════╤══════════╤══════════╤══════════╤════════════════╕
│   n_clusters │   vector_size │   window │   epochs │      NMI │ running_time   │
╞══════════════╪═══════════════╪══════════╪══════════╪══════════╪════════════════╡
│          100 │           200 │       10 │       10 │ 0.188465 │ 0:00:03.302823 │
├──────────────┼───────────────┼──────────┼──────────┼──────────┼────────────────┤
│           20 │           200 │       10 │       10 │ 0.132166 │ 0:00:00.626210 │
├──────────────┼───────────────┼──────────┼──────────┼──────────┼────────────────┤
│           50 │           200 │       10 │       10 │ 0.16561  │ 0:00:01.538886 │
├──────────────┼───────────────┼──────────┼──────────┼──────────┼────────────────┤
│          300 │           200 │       10 │       10 │ 0.208517 │ 0:00:08.608336 │
╘══════════════╧═══════════════╧══════════╧══════════╧══════════╧════════════════╛
